In [4]:
#batch_id = "MANUAL"
#api_base_url = "http://localhost:8000"
#api_client_id = "fabric"
#api_client_secret = "fabric-dev-secret"
#entities = "[]"

StatementMeta(, fc0fa34d-c8bf-4b32-99a6-c2411ca339b8, 6, Finished, Available, Finished, False)

In [5]:
# Fabric's Copy activity paginates this API correctly — the API's own request
# log showed 547 cursor-bearing requests — but buffers the whole response into
# a single write, so only one page survives. Table sink, file sink, filename
# expressions and folder paths all behave identically. Doing the HTTP here
# keeps every page while staying inside the control framework.
#
# All entities are handled in one session rather than one notebook activity
# per entity: each activity costs a fresh Spark session, and four of those
# exhausts a trial capacity before any data moves.
import json, time, urllib.request, urllib.parse
from datetime import datetime, timezone
from pyspark.sql import functions as F

run_ts = datetime.now(timezone.utc)
entity_list = json.loads(entities) if isinstance(entities, str) else entities
if not entity_list:
    raise ValueError("No entities passed — check the entities parameter uses @string(...)")

def call(req, attempts=5):
    """Tunnels and rate limits both produce transient failures. A 98-page
    pull should not die on one of them."""
    for i in range(attempts):
        try:
            return json.load(urllib.request.urlopen(req, timeout=120))
        except Exception as e:
            if i == attempts - 1:
                raise RuntimeError(f"failed after {attempts} attempts — {e}")
            print(f"    retry {i+1}: {e}")
            time.sleep(2 ** i)

# One token for the whole run. It lasts 15 minutes, comfortably longer than
# four entities take.
tok = call(urllib.request.Request(
    api_base_url + "/oauth/token",
    data=urllib.parse.urlencode({"grant_type": "client_credentials",
                                 "client_id": api_client_id,
                                 "client_secret": api_client_secret}).encode()
))["access_token"]
print(f"token acquired — {len(entity_list)} entities to pull")

StatementMeta(, fc0fa34d-c8bf-4b32-99a6-c2411ca339b8, 7, Finished, Available, Finished, False)

medication-orders: 97,511 records across 98 pages


In [6]:
results = []

for ent in entity_list:
    source_object   = ent["source_object"]
    target_table    = ent["target_table"]
    source_system   = ent["source_system"]
    entity_name     = ent["entity_name"]
    load_type       = ent.get("load_type", "incremental")
    watermark_value = ent.get("watermark_value", "1900-01-01 00:00:00")
    label = f"{source_system}.{entity_name}"

    qs = "limit=1000"
    if load_type == "incremental":
        qs += "&updated_since=" + urllib.parse.quote(watermark_value.replace(" ", "T"))

    rows, url, pages = [], f"{api_base_url}/api/v1/{source_object}?{qs}", 0
    while url:
        r = urllib.request.Request(url)
        r.add_header("Authorization", "Bearer " + tok)
        page = call(r)
        rows.extend(page["data"])
        pages += 1
        cur = page["pagination"]["next_cursor"]
        url = f"{api_base_url}/api/v1/{source_object}?{qs}&cursor={cur}" if cur else None

    if not rows:
        print(f"  {label:<28} no new records")
        results.append({"entity": label, "rows": 0, "pages": pages})
        continue

    df = spark.createDataFrame(rows)
    payload = df.columns
    df = (df.withColumn("_source_system", F.lit(source_system))
            .withColumn("_entity_name", F.lit(entity_name))
            .withColumn("_batch_id", F.lit(batch_id))
            .withColumn("_ingest_ts", F.lit(run_ts))
            .withColumn("_load_date", F.lit(run_ts.date()))
            .withColumn("_row_hash", F.sha2(F.concat_ws("||", *[
                F.coalesce(F.upper(F.trim(F.col(c).cast("string"))), F.lit("<NULL>"))
                for c in payload]), 256)))

    # full_snapshot reloads everything each run, so replace today's partition
    # rather than appending a duplicate copy.
    if load_type == "full_snapshot" and spark.catalog.tableExists(target_table):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"_load_date = '{run_ts.date()}'")
           .option("mergeSchema", "true").partitionBy("_load_date")
           .saveAsTable(target_table))
    else:
        (df.write.format("delta").mode("append")
           .option("mergeSchema", "true").partitionBy("_load_date")
           .saveAsTable(target_table))

    n = df.count()
    print(f"  {label:<28} {n:>8,} rows  ({pages} pages)")
    results.append({"entity": label, "rows": n, "pages": pages})

total = sum(r["rows"] for r in results)
print(f"\nTotal: {total:,} rows across {len(results)} entities")

mssparkutils.notebook.exit(json.dumps({
    "batch_id": batch_id,
    "entities": len(results),
    "total_rows": total,
}))

StatementMeta(, fc0fa34d-c8bf-4b32-99a6-c2411ca339b8, 8, Finished, Available, Finished, False)

pharm_medication_order: 97,511 rows written
ExitValue: {"batch_id": "MANUAL", "rows_written": 97511, "pages": 98}